# Customer Churn Analysis & Model Training

This notebook covers the Exploratory Data Analysis (EDA), Preprocessing, and Model Training for the Customer Churn Prediction project. We implement a custom Logistic Regression model from scratch using NumPy and compare it with the scikit-learn implementation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append('../src')

from data_preprocessing import preprocess_data, handle_imbalance
from logistic_regression import LogisticRegression
from evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_pr_curve, plot_loss_curve
from utils import save_object, load_object
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import f1_score

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Churn Distribution
sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.show()

# Churn Rate by Contract
sns.barplot(data=df, x='Contract', y=(df['Churn']=='Yes').astype(int))
plt.title('Churn Rate by Contract Type')
plt.show()

## 3. Preprocessing & Imbalance Handling

In [ ]:
X, y, preprocessor = preprocess_data(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Original training shape: {X_train.shape}")
X_train_res, y_train_res = handle_imbalance(X_train, y_train)
print(f"Resampled training shape: {X_train_res.shape}")

## 4. Custom Model Training

In [ ]:
custom_model = LogisticRegression(learning_rate=0.5, n_iterations=1000, lambda_=0.1)
custom_model.fit(X_train_res.values, y_train_res.values)

plot_loss_curve(custom_model.loss_history)

## 5. Evaluation & Comparison

In [ ]:
y_prob = custom_model.predict_proba(X_test.values)
threshold = 0.46 # Pre-calculated optimal threshold
y_pred = (y_prob >= threshold).astype(int)

evaluate_model(y_test, y_pred, y_prob, title="Custom LR")
plot_confusion_matrix(y_test, y_pred)